# What does the client's scoring rule do with a policy?

This notebook loads the bookings, adds the two columns the scoring rule needs, and scores the simplest policy there is: sell nothing extra. It runs in under a minute. Change nothing here; your team's work goes in `src/`.

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

import contract
import data

In [2]:
bookings = data.read_raw()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
print(f"{len(bookings):,} bookings, arrivals {bookings.arrival_date.min().date()} to {bookings.arrival_date.max().date()}")

119,390 bookings, arrivals 2015-07-01 to 2017-08-31


In [3]:
train, choose, proof = contract.split_periods(bookings)
for name, part in [("train", train), ("choose", choose), ("proof", proof)]:
    print(f"{name:>6}: {len(part):>7,} bookings")

 train:  49,157 bookings
choose:  29,546 bookings
 proof:  40,687 bookings


## What does selling nothing extra cost on the choose period?

Every cancellation leaves a room empty and nobody is walked. This is a floor, not the client's rule: the client already oversells with a flat rate, and your savings are measured against that.

In [4]:
nights = contract.nightly_table(choose, prob=np.zeros(len(choose)))
nights.head()

book  cancelled  expected  spread  stay_value
hotel      arrival_date                                               
City Hotel 2016-07-01     141         68       0.0     0.0  319.747589
           2016-07-02      80         33       0.0     0.0  397.106875
           2016-07-03      85         21       0.0     0.0  478.769529
           2016-07-04      92         30       0.0     0.0  440.365543
           2016-07-05      74         14       0.0     0.0  405.705811

In [5]:
contract.night_cost(nights, oversell=np.zeros(len(nights)))

{'cost_eur': 4155340.86,
 'empty_rooms': 10894,
 'walked_guests': 0,
 'walks_per_1000': 0.0,
 'ceiling_met': True,
 'nights': 368}

## What comes next?

- Check every number in `BRIEF.md` against `bookings`, and record each one in the charter.
- Formalise the client's flat rule in `src/backtest.py` and score it on these same nights.
- Leave the proof period alone until every setting is frozen.

In [6]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent      
DATA = ROOT / 'data'   
df = pd.read_csv(DATA / 'hotels.csv')   

In [7]:
df.sample(20)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
82355,City Hotel,1,94,2016,March,10,4,0,2,2,...,Non Refund,154.0,NaN,28,Transient,80.00,0,0,Canceled,2015-12-29
73659,City Hotel,1,127,2017,August,35,29,0,3,3,...,No Deposit,9.0,NaN,0,Transient,195.00,0,1,Canceled,2017-06-21
20455,Resort Hotel,0,1,2016,January,5,28,0,1,1,...,No Deposit,250.0,NaN,0,Transient,45.00,1,0,Check-Out,2016-01-29
44058,City Hotel,0,83,2015,September,40,30,0,1,2,...,No Deposit,26.0,NaN,77,Transient-Party,112.20,0,0,Check-Out,2015-10-01
89872,City Hotel,0,14,2016,May,22,27,0,1,1,...,No Deposit,21.0,NaN,0,Transient-Party,62.50,0,0,Check-Out,2016-05-28
100789,City Hotel,0,226,2016,October,44,29,2,2,2,...,No Deposit,3.0,NaN,0,Transient-Party,70.00,0,1,Check-Out,2016-11-02
65782,City Hotel,1,60,2017,April,14,8,2,1,2,...,No Deposit,9.0,NaN,0,Transient,126.00,0,1,Canceled,2017-03-05
38171,Resort Hotel,0,130,2017,July,28,9,2,0,2,...,No Deposit,242.0,NaN,0,Transient-Party,140.00,0,1,Check-Out,2017-07-11
93453,City Hotel,0,4,2016,July,29,15,2,5,2,...,No Deposit,89.0,NaN,0,Transient,133.41,0,1,Check-Out,2016-07-22
2676,Resort Hotel,0,120,2015,October,43,24,4,6,2,...,No Deposit,175.0,NaN,0,Transient,34.56,0,0,Check-Out,2015-11-03


In [8]:
df.columns.to_list

<bound method IndexOpsMixin.tolist of Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='str')>

In [9]:
df["arrival_date"] = data.arrival_date(df)
df["nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

In [10]:
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,nights
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,1
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,1
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03,2015-07-01,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06,2017-08-30,7
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07,2017-08-31,7
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07,2017-08-31,7
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07,2017-08-31,7


### BIN A


In [11]:
bin_A= df[["hotel","lead_time",
           "arrival_date",
           "arrival_date_week_number", 
           "is_repeated_guest", 
           "previous_cancellations",
           "previous_bookings_not_canceled",
           "country", "market_segment", "distribution_channel",
           "agent", "company", "customer_type"]]

In [12]:
bin_A

,hotel,lead_time,arrival_date,arrival_date_week_number,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,country,market_segment,distribution_channel,agent,company,customer_type
0,Resort Hotel,342,2015-07-01,27,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient
1,Resort Hotel,737,2015-07-01,27,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient
2,Resort Hotel,7,2015-07-01,27,0,0,0,GBR,Direct,Direct,NaN,NaN,Transient
3,Resort Hotel,13,2015-07-01,27,0,0,0,GBR,Corporate,Corporate,304.0,NaN,Transient
4,Resort Hotel,14,2015-07-01,27,0,0,0,GBR,Online TA,TA/TO,240.0,NaN,Transient
...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,23,2017-08-30,35,0,0,0,BEL,Offline TA/TO,TA/TO,394.0,NaN,Transient
119386,City Hotel,102,2017-08-31,35,0,0,0,FRA,Online TA,TA/TO,9.0,NaN,Transient
119387,City Hotel,34,2017-08-31,35,0,0,0,DEU,Online TA,TA/TO,9.0,NaN,Transient
119388,City Hotel,109,2017-08-31,35,0,0,0,GBR,Online TA,TA/TO,89.0,NaN,Transient


## Prediction

In [13]:
y = df["is_canceled"]

In [14]:
mvp_col=pd.concat([bin_A,y],axis=1)

In [15]:
mvp_col

,hotel,lead_time,arrival_date,arrival_date_week_number,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,country,market_segment,distribution_channel,agent,company,customer_type,is_canceled
0,Resort Hotel,342,2015-07-01,27,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
1,Resort Hotel,737,2015-07-01,27,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
2,Resort Hotel,7,2015-07-01,27,0,0,0,GBR,Direct,Direct,NaN,NaN,Transient,0
3,Resort Hotel,13,2015-07-01,27,0,0,0,GBR,Corporate,Corporate,304.0,NaN,Transient,0
4,Resort Hotel,14,2015-07-01,27,0,0,0,GBR,Online TA,TA/TO,240.0,NaN,Transient,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,23,2017-08-30,35,0,0,0,BEL,Offline TA/TO,TA/TO,394.0,NaN,Transient,0
119386,City Hotel,102,2017-08-31,35,0,0,0,FRA,Online TA,TA/TO,9.0,NaN,Transient,0
119387,City Hotel,34,2017-08-31,35,0,0,0,DEU,Online TA,TA/TO,9.0,NaN,Transient,0
119388,City Hotel,109,2017-08-31,35,0,0,0,GBR,Online TA,TA/TO,89.0,NaN,Transient,0


In [16]:
duplicate=mvp_col[mvp_col.duplicated(keep=False)]

In [17]:
duplicate

,hotel,lead_time,arrival_date,arrival_date_week_number,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,country,market_segment,distribution_channel,agent,company,customer_type,is_canceled
4,Resort Hotel,14,2015-07-01,27,0,0,0,GBR,Online TA,TA/TO,240.0,NaN,Transient,0
5,Resort Hotel,14,2015-07-01,27,0,0,0,GBR,Online TA,TA/TO,240.0,NaN,Transient,0
21,Resort Hotel,72,2015-07-01,27,0,0,0,PRT,Direct,Direct,250.0,NaN,Transient,0
22,Resort Hotel,72,2015-07-01,27,0,0,0,PRT,Direct,Direct,250.0,NaN,Transient,0
23,Resort Hotel,72,2015-07-01,27,0,0,0,PRT,Direct,Direct,250.0,NaN,Transient,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119362,City Hotel,198,2017-08-29,35,0,0,0,ITA,Online TA,TA/TO,9.0,NaN,Transient,0
119371,City Hotel,175,2017-08-31,35,0,0,0,NLD,Offline TA/TO,TA/TO,42.0,NaN,Transient,0
119372,City Hotel,175,2017-08-31,35,0,0,0,NLD,Offline TA/TO,TA/TO,42.0,NaN,Transient,0
119373,City Hotel,175,2017-08-31,35,0,0,0,NLD,Offline TA/TO,TA/TO,42.0,NaN,Transient,0


In [18]:
all_cols=list(mvp_col.columns)
dups = mvp_col[mvp_col.duplicated(keep=False)]
dups = dups.sort_values(all_cols)
dups.head(20)

,hotel,lead_time,arrival_date,arrival_date_week_number,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,country,market_segment,distribution_channel,agent,company,customer_type,is_canceled
40394,City Hotel,0,2015-07-24,30,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40395,City Hotel,0,2015-07-24,30,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40396,City Hotel,0,2015-07-24,30,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40397,City Hotel,0,2015-07-24,30,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40398,City Hotel,0,2015-07-24,30,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40533,City Hotel,0,2015-07-28,31,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40534,City Hotel,0,2015-07-28,31,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40536,City Hotel,0,2015-07-28,31,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40537,City Hotel,0,2015-07-28,31,0,0,0,PRT,Direct,Direct,NaN,NaN,Transient,0
40542,City Hotel,0,2015-07-29,31,0,0,0,PRT,Complementary,TA/TO,NaN,NaN,Transient,0


In [19]:
sizes = mvp_col.groupby(all_cols, dropna=False).size()
#print(sizes[sizes > 1].describe())
print(sizes[sizes > 1].value_counts().head(10))   

2     8288
3     1594
4      620
5      335
6      203
10     126
7      117
8      107
20     107
9       99
Name: count, dtype: int64
